In [ ]:
# Cell 1：安装依赖
%pip install -q openai datasets pandas tqdm numpy

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = ""

In [ ]:
# Cell 2：初始化
import os
import re
import string
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
from openai import OpenAI

# 先在环境变量中设置 OPENAI_API_KEY
# PowerShell: $env:OPENAI_API_KEY = "sk-..."

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

CHAT_MODEL = "gpt-4o"
EMBEDDING_MODEL = "text-embedding-3-small"

dataset = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
NUM_SAMPLES = 200
demo_data = dataset.shuffle(seed=42).select(range(NUM_SAMPLES))

In [ ]:
# Cell 3：将每个候选段落转成 RAG 文档块
def build_documents(context):
    documents = []

    for title, sentences in zip(context["title"], context["sentences"]):
        text = " ".join(sentences).strip()
        if text:
            documents.append({
                "title": title,
                "text": text,
                "content": f"标题：{title}\n内容：{text}"
            })

    return documents

documents = build_documents(demo_data[0]["context"])
print(documents[0]["content"][:500])

标题：Robin Barton
内容：Robin Barton (born 5 November 1958) is a British art dealer dealing primarily with Banksy's.  Barton studied photography and graphic design at the Exeter College of Art and Design and this was his first encounter with Russell Young.  Moving to London in 1980 he began working as a freelance photographer for music and fashion publications "Sounds", "NME", "Blitz", "The Face" moving on to working regularly for pioneering "Independent Magazine" photographing amongst others Sir Ale


In [ ]:
# Cell 4：Embedding、余弦相似度与检索函数
def get_embeddings(texts):
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts
    )
    return np.array([item.embedding for item in response.data], dtype=np.float32)

def cosine_similarity(query_vector, document_vectors):
    query_vector = query_vector / np.linalg.norm(query_vector)
    document_vectors = document_vectors / np.linalg.norm(
        document_vectors, axis=1, keepdims=True
    )
    return document_vectors @ query_vector

def retrieve(question, documents, top_k=3):
    # 简单 demo：为当前问题候选段落生成 embedding。
    # 生产场景应将 document_embeddings 持久化至向量数据库。
    document_embeddings = get_embeddings([doc["content"] for doc in documents])
    question_embedding = get_embeddings([question])[0]

    scores = cosine_similarity(question_embedding, document_embeddings)
    top_indices = np.argsort(scores)[::-1][:top_k]

    retrieved_docs = []
    for idx in top_indices:
        doc = documents[idx].copy()
        doc["score"] = float(scores[idx])
        retrieved_docs.append(doc)

    return retrieved_docs

In [ ]:
def rag_answer(question, documents, top_k=3):
    retrieved_docs = retrieve(question, documents, top_k=top_k)

    retrieved_context = "\n\n".join(
        f"[Evidence {i + 1}]\nTitle: {doc['title']}\nContent: {doc['text']}"
        for i, doc in enumerate(retrieved_docs)
    )

    prompt = f"""You are a careful multi-hop question-answering assistant.

Answer the question using only the retrieved evidence below.
Do not use outside knowledge.
Return only a short final answer, without explaining your reasoning.
If the evidence is insufficient, return exactly:
Cannot be determined from the provided context.

Question:
{question}

Retrieved Evidence:
{retrieved_context}
"""

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You answer questions strictly using the provided retrieved "
                    "evidence. Do not add external knowledge."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        temperature=0,
        max_tokens=100,
    )

    answer = response.choices[0].message.content.strip()
    return answer, retrieved_docs

In [ ]:
# Cell 6：运行 RAG Demo
results = []

for item in tqdm(demo_data, desc="RAG + GPT-4o"):
    documents = build_documents(item["context"])

    prediction, retrieved_docs = rag_answer(
        question=item["question"],
        documents=documents,
        top_k=5
    )

    results.append({
        "id": item["id"],
        "question": item["question"],
        "gold_answer": item["answer"],
        "prediction": prediction,
        "retrieved_titles": [doc["title"] for doc in retrieved_docs],
        "retrieval_scores": [round(doc["score"], 3) for doc in retrieved_docs],
    })

df = pd.DataFrame(results)
display(df)

RAG + GPT-4o:   0%|          | 0/200 [00:00<?, ?it/s]

,id,question,gold_answer,prediction,retrieved_titles,retrieval_scores
0,5add1d575542992c1e3a2540,What nationality was Oliver Reed's character i...,Prussian,"Oliver Reed's character in the film ""Royal Fla...","[Oliver Reed, Royal Flash (film), Royal Flash,...","[0.629, 0.618, 0.471, 0.452, 0.407]"
1,5ac55ea55542993e66e82377,Pacific Mozart Ensemble performed which German...,Kurt Julian Weill,Kurt Weill,"[The Flight Across the Ocean, Pacific Mozart E...","[0.563, 0.476, 0.404, 0.393, 0.383]"
2,5ac3a76e554299741d48a2be,"Who released the song ""With or Without You"" fi...",U2,"U2 released the song ""With or Without You"" first.","[With or Without You, Believe (Jai McDowall al...","[0.688, 0.527, 0.471, 0.447, 0.328]"
3,5a7133565542994082a3e65c,"What Kentucky county has a population of 60,31...",Oldham County,Oldham County,"[Lake Louisvilla, Louisville, Oldham County, K...","[0.71, 0.524, 0.475, 0.44, 0.326]"
4,5ac1a4745542991316484b82,"Para Hills West, South Australia lies within a...","138,535","138,535","[Para Hills West, South Australia, Para Hills,...","[0.676, 0.625, 0.514, 0.504, 0.482]"
...,...,...,...,...,...,...
195,5a7642f7554299109176e6e8,who is younger Ulrich Walter or Luca Parmitano?,Luca Parmitano,Luca Parmitano is younger.,"[Ulrich Walter, Luca Parmitano, Otto F. Walter...","[0.522, 0.521, 0.45, 0.425, 0.355]"
196,5a756b175542996c70cfaef7,"Which peak stands higher, Muztagh Ata or Manaslu?",Manaslu,Manaslu stands higher at 8163 meters.,"[Muztagh Ata, Manaslu, Lenin Peak, Ngadi Chuli...","[0.667, 0.638, 0.525, 0.524, 0.405]"
197,5a83161555429940e5e1a958,What age did a wrestler began his training at ...,15,15,"[Chuck Taylor (wrestler), Ricochet (wrestler),...","[0.519, 0.459, 0.344, 0.333, 0.272]"
198,5ab3eac5554299753aec59ec,Which season did Brain Billick coach the Balti...,6th season,The 2001 season.,"[Brian Billick, 2001 Baltimore Ravens season, ...","[0.71, 0.704, 0.691, 0.546, 0.541]"


In [ ]:
# Cell 7：简易 EM / F1 评测
def normalize_answer(text):
    text = text.lower()
    text = "".join(ch for ch in text if ch not in string.punctuation)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())

def exact_match(prediction, gold):
    return int(normalize_answer(prediction) == normalize_answer(gold))

def token_f1(prediction, gold):
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(gold).split()

    common = Counter(pred_tokens) & Counter(gold_tokens)
    overlap = sum(common.values())

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

df["EM"] = df.apply(lambda x: exact_match(x.prediction, x.gold_answer), axis=1)
df["F1"] = df.apply(lambda x: token_f1(x.prediction, x.gold_answer), axis=1)

display(df[["question", "gold_answer", "prediction", "retrieved_titles", "EM", "F1"]])
print(f"average EM: {df['EM'].mean():.3f}")
print(f"average F1: {df['F1'].mean():.3f}")

,question,gold_answer,prediction,retrieved_titles,EM,F1
0,What nationality was Oliver Reed's character i...,Prussian,Cannot be determined from the provided context.,"[Oliver Reed, Royal Flash (film), Royal Flash]",0,0.000000
1,Pacific Mozart Ensemble performed which German...,Kurt Julian Weill,Kurt Weill,"[The Flight Across the Ocean, Pacific Mozart E...",0,0.800000
2,"Who released the song ""With or Without You"" fi...",U2,"U2 released the song ""With or Without You"" first.","[With or Without You, Believe (Jai McDowall al...",0,0.222222
3,"What Kentucky county has a population of 60,31...",Oldham County,Oldham County,"[Lake Louisvilla, Louisville, Oldham County, K...",1,1.000000
4,"Para Hills West, South Australia lies within a...","138,535","138,535","[Para Hills West, South Australia, Para Hills,...",1,1.000000
...,...,...,...,...,...,...
195,who is younger Ulrich Walter or Luca Parmitano?,Luca Parmitano,Luca Parmitano is younger.,"[Ulrich Walter, Luca Parmitano, Otto F. Walter]",0,0.666667
196,"Which peak stands higher, Muztagh Ata or Manaslu?",Manaslu,Manaslu stands higher.,"[Muztagh Ata, Manaslu, Lenin Peak]",0,0.500000
197,What age did a wrestler began his training at ...,15,15,"[Chuck Taylor (wrestler), Ricochet (wrestler),...",1,1.000000
198,Which season did Brain Billick coach the Balti...,6th season,The third season.,"[Brian Billick, 2001 Baltimore Ravens season, ...",0,0.500000


average EM: 0.340
average F1: 0.498


In [ ]:
def build_full_context(context):
    passages = []

    for title, sentences in zip(context["title"], context["sentences"]):
        text = " ".join(sentences).strip()
        if text:
            passages.append(f"Title: {title}\nContent: {text}")

    return "\n\n".join(passages)

In [ ]:
# Cell 10：无 RAG —— 直接基于完整上下文回答
def no_rag_answer(question, context):
    full_context = build_full_context(context)

    prompt = f"""You are a careful multi-hop question-answering assistant.

Answer the question using only the full context below.
Do not use outside knowledge.
Return only a short final answer, without explaining your reasoning.
If the context is insufficient, return exactly:
Cannot be determined from the provided context.

Question:
{question}

Full Context:
{full_context}
"""

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You answer questions strictly using the provided context. "
                    "Do not add external knowledge."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        temperature=0,
        max_tokens=100,
    )

    return response.choices[0].message.content.strip()

In [ ]:
# Cell 10：无 RAG —— 直接基于完整上下文回答
def no_rag_answer(question, context):
    full_context = build_full_context(context)

    prompt = f"""You are a careful multi-hop question-answering assistant.

Answer the question using only the full context below.
Do not use outside knowledge.
Return only a short final answer, without explaining your reasoning.
If the context is insufficient, return exactly:
Cannot be determined from the provided context.

Question:
{question}

Full Context:
{full_context}
"""

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You answer questions strictly using the provided context. "
                    "Do not add external knowledge."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        temperature=0,
        max_tokens=100,
    )

    return response.choices[0].message.content.strip()

In [ ]:
# Cell 11：在与 RAG 完全相同的 demo_data 上评测无 RAG
no_rag_results = []

for item in tqdm(demo_data, desc="No-RAG + GPT-4o"):
    prediction = no_rag_answer(
        question=item["question"],
        context=item["context"],
    )

    no_rag_results.append({
        "id": item["id"],
        "question": item["question"],
        "gold_answer": item["answer"],
        "no_rag_prediction": prediction,
    })

no_rag_df = pd.DataFrame(no_rag_results)

no_rag_df["no_rag_EM"] = no_rag_df.apply(
    lambda x: exact_match(x["no_rag_prediction"], x["gold_answer"]), axis=1
)
no_rag_df["no_rag_F1"] = no_rag_df.apply(
    lambda x: token_f1(x["no_rag_prediction"], x["gold_answer"]), axis=1
)

print(f"No-RAG average EM: {no_rag_df['no_rag_EM'].mean():.3f}")
print(f"No-RAG average F1: {no_rag_df['no_rag_F1'].mean():.3f}")

No-RAG + GPT-4o:   0%|          | 0/200 [00:00<?, ?it/s]

No-RAG average EM: 0.490
No-RAG average F1: 0.653


In [ ]:
# Cell 9：No-RAG / No-Context baseline
def direct_answer(question):
    prompt = f"""Answer the following question concisely.

Return only the final answer, without explanation.

Question:
{question}
"""

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are a helpful question-answering assistant.",
            },
            {"role": "user", "content": prompt},
        ],
        temperature=0,
        max_tokens=100,
    )

    return response.choices[0].message.content.strip()

In [ ]:
# Cell 10：评测 Direct LLM（没有 RAG、没有上下文）
direct_results = []

for item in tqdm(demo_data, desc="Direct GPT-4o"):
    prediction = direct_answer(item["question"])

    direct_results.append({
        "id": item["id"],
        "question": item["question"],
        "gold_answer": item["answer"],
        "direct_prediction": prediction,
    })

direct_df = pd.DataFrame(direct_results)

direct_df["direct_EM"] = direct_df.apply(
    lambda x: exact_match(x["direct_prediction"], x["gold_answer"]), axis=1
)
direct_df["direct_F1"] = direct_df.apply(
    lambda x: token_f1(x["direct_prediction"], x["gold_answer"]), axis=1
)

print(f"Direct GPT-4o average EM: {direct_df['direct_EM'].mean():.3f}")
print(f"Direct GPT-4o average F1: {direct_df['direct_F1'].mean():.3f}")

Direct GPT-4o:   0%|          | 0/200 [00:00<?, ?it/s]

Direct GPT-4o average EM: 0.360
Direct GPT-4o average F1: 0.494
